# /extract — Endpoint Evaluation

For each test article shows:
- **Original text** (full)
- **Extracted text** returned by the endpoint
- Which sentences were kept / dropped
- Compression ratio and embedding shape

**Prerequisite:** NLP service running. `/readyz` → 200.

In [13]:
import sys, time, re, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, NLP_BASE_URL, HEADERS

cases = load_fixture('summarize_cases.json')
print(f'Loaded {len(cases)} test cases')

Loaded 5 test cases


In [14]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, f'Service not ready: {r.status_code} {r.text}'
print('Service ready')

Service ready


In [15]:
def split_sentences(text: str) -> list[str]:
    """Naive sentence splitter on . ! ? followed by whitespace or end."""
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]


def print_diff(original: str, extract: str) -> None:
    orig_sents = split_sentences(original)
    ext_lower  = extract.lower()

    print('  ORIGINAL SENTENCES:')
    for i, s in enumerate(orig_sents, 1):
        # mark as kept if the sentence (or most of it) appears in the extract
        kept = s[:40].lower() in ext_lower
        marker = '  ✓' if kept else '  ✗'
        print(f'{marker} [{i}] {s}')

    ratio = len(extract) / max(len(original), 1) * 100
    print()
    print(f'  EXTRACT ({len(extract)} chars  /  {ratio:.0f}% of original):')
    print(f'  {extract}')

In [16]:
EMBEDDING_DIM = 384
results = []

for case in cases:
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/extract',
        json={'article_id': case['article_id'], 'text': case['text']},
        headers=HEADERS,
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, \
        f"{case['article_id']}: HTTP {resp.status_code} — {resp.text}"
    data = resp.json()

    extract      = data['extract']
    embedding    = data['embedding_raw']
    extract_ok   = bool(extract.strip())
    embed_ok     = len(embedding) == EMBEDDING_DIM
    passed       = extract_ok and embed_ok
    ratio        = len(extract) / max(len(case['text']), 1) * 100

    results.append({
        'id':           case['article_id'],
        'extract_ok':   extract_ok,
        'embed_ok':     embed_ok,
        'embed_dim':    len(embedding),
        'extract_len':  len(extract),
        'orig_len':     len(case['text']),
        'ratio_pct':    ratio,
        'latency_s':    latency,
        'pass':         passed,
        'extract':      extract,
        'embedding_raw': embedding,
    })

    icon = '✅' if passed else '❌'
    print(f"{icon} [{case['article_id']}]  {case.get('description', '')}")
    print(f"   embed_dim={len(embedding)}  latency={latency:.2f}s")
    print()
    print_diff(case['text'], extract)
    print()
    print('─' * 72)
    print()

✅ [sum-001]  Standard in-scope cycling infrastructure article
   embed_dim=384  latency=0.94s

  ORIGINAL SENTENCES:
  ✓ [1] El Ayuntamiento de Madrid ha aprobado esta semana la ampliación del carril bici en la Gran Vía, una de las arterias principales de la capital.
  ✓ [2] La nueva infraestructura ciclista se extenderá desde la plaza de España hasta la calle Alcalá, con una longitud total de 1,3 kilómetros.
  ✓ [3] Las obras, que comenzarán el próximo mes de junio, tienen un presupuesto de 2,4 millones de euros y se prolongarán durante aproximadamente tres semanas.
  ✓ [4] El concejal de Movilidad Sostenible ha destacado que este proyecto forma parte del Plan de Movilidad Urbana 2025-2030, que prevé la construcción de 150 kilómetros adicionales de carril bici en los próximos cinco años.
  ✓ [5] La medida ha sido recibida con satisfacción por las asociaciones ciclistas de la ciudad, aunque algunos vecinos han expresado preocupación por la reducción de plazas de aparcamiento.

  EXTRAC

In [17]:
passing    = [r for r in results if r['pass']]
avg_lat    = sum(r['latency_s']   for r in results) / len(results)
avg_ratio  = sum(r['ratio_pct']   for r in results) / len(results)
avg_elen   = sum(r['extract_len'] for r in results) / len(results)

print_scorecard('/extract', {
    'Cases':                        len(results),
    'Passing (extract + embed OK)': f'{len(passing)}/{len(results)}',
    'Expected embedding dim':       EMBEDDING_DIM,
    'Avg extract length (chars)':   avg_elen,
    'Avg compression ratio (%)':    avg_ratio,
    'Avg latency (s)':              avg_lat,
})


  /extract
  Cases                               5
  Passing (extract + embed OK)        5/5
  Expected embedding dim              384
  Avg extract length (chars)          696.200
  Avg compression ratio (%)           89.488
  Avg latency (s)                     0.322

